# Words

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("..\\.."))
from Modules.latentspacemodel import *
from Modules.inference import *
from Modules.afteranalyses import *

## Network

In [ ]:
# Cargar la red "Words/Ladder Graph"

from string import ascii_lowercase as lowercase
def generate_graph(words):
    G = nx.Graph(name="words")
    lookup = {c: lowercase.index(c) for c in lowercase}

    def edit_distance_one(word):
        for i in range(len(word)):
            left, c, right = word[0:i], word[i], word[i + 1:]
            j = lookup[c]
            for cc in lowercase[j + 1:]:
                yield left + cc + right

    candgen = (
        (word, cand)
        for word in sorted(words)
        for cand in edit_distance_one(word)
        if cand in words
    )

    G.add_nodes_from(words)

    for word, cand in candgen:
        G.add_edge(word, cand)

    return G


def words_graph():
    words = set()
    with open("C://Users//carlo//OneDrive//Documents//Latentspacemodels_Networks//Complement//Examples//Words//words_dat.txt", "r", encoding="utf-8") as fh:
        for line in fh:
            if line.startswith("*"):
                continue
            w = line[0:5]
            words.add(w.strip())
    return generate_graph(words)


print("Complete graph")
G = words_graph()
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))

# Giant component
print("Giant component")
largest_cc = max(nx.connected_components(G), key=len)
G = G.subgraph(largest_cc).copy()
print("Nodes in giant component:", G.number_of_nodes())
print("Edges in giant component:", G.number_of_edges())


In [ ]:
pos = nx.spring_layout(G, seed=80)
plt.figure(figsize=(10,10))
nx.draw(G, pos=pos, with_labels=True, node_color="lightblue", edge_color="gray", node_size=30, font_size=2)
plt.show()

node_mapping = {node: i for i, node in enumerate(G.nodes())}
G = nx.relabel_nodes(G, node_mapping)
pos_relabel = {node_mapping[k]: v for k, v in pos.items()}
plt.figure(figsize=(10,10))
nx.draw(G,pos=pos_relabel, with_labels=True, node_color="lightblue", edge_color="gray", node_size=30, font_size=2)
plt.show()

inverse_mapping = {v: k for k, v in node_mapping.items()}

Y = nx.to_numpy_array(G, dtype=int)
Y = (Y > 0).astype(int)
n = Y.shape[0]
plt.figure(figsize=(10,10))
sns.heatmap(Y[0:30,0:30], annot=False, cmap="Blues", cbar=False, square=True, linewidths=0.1, linecolor="white")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.gca().xaxis.tick_top()
plt.gca().tick_params(top=True, bottom=False, labeltop=True, labelbottom=False, labelsize=6)
plt.show()

In [ ]:
seed_MAP = 42
diam = 10.0
n = Y.shape[0]
lr_MAP = 1e-2
n_starts_MAP = 1
n_iter_MAP = 25000
n_sim_iv = 2500
n_iter_cv = 1000
n_folds_cv = 5

## Models over $\mathbb{R}^d$

### Model $\mathbb{R^1}$

#### Without weights

In [ ]:
geometry = EuclideanGeometry(d=1, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R1_c.pkl")

#### With weights

In [ ]:
geometry = EuclideanGeometry(d=1, D=diam)
alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R1_p.pkl")

### Model $\mathbb{R}^2$

#### Without weigths

In [ ]:
geometry = EuclideanGeometry(d=2, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R2_c.pkl")

#### With weights

In [ ]:
geometry = EuclideanGeometry(d=2, D = diam)
alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R2_p.pkl")

### Model $\mathbb{R}^3$

#### Without weights

In [ ]:
geometry = EuclideanGeometry(d=3, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R3_c.pkl")

#### With weights

In [ ]:
geometry = EuclideanGeometry(d=3, D = diam)
alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_R3_p.pkl")

## Models over $\mathbb{S}^d$

### Model $\mathbb{S}^1$

#### Without weights

In [ ]:
with open("Florentine_R2_c.pkl", "rb") as f:
    results_R1_c = pickle.load(f)

Z_init = np.zeros((15,2))
Z_init[:,:] = np.asarray(results_R1_c['latent_params']['Z'])
for i in range(15):
    Z_init[i] = geometry.project_to_domain(Z_init[i])

In [ ]:
geometry = SphericalGeometry(d=1, D=diam)
with open("Florentine_R2_c.pkl", "rb") as f:
    results_R2_c = pickle.load(f)
Z_init = np.zeros((15,2))
Z_init[:,:] = np.asarray(results_R2_c['latent_params']['Z'])
for i in range(15):
    Z_init[i] = geometry.project_to_domain(Z_init[i])
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True, init_params = {"alpha0":0.0,"Z": Z_init})

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_S1_c.pkl")

alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]

#### With weights

In [ ]:
geometry = SphericalGeometry(d=1, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_S1_p.pkl")

### Model $\mathbb{S}^2$

#### Without weights

In [ ]:
with open("Florentine_R3_p.pkl", "rb") as f:
    results_R3_p = pickle.load(f)

Z_init = np.zeros((n,3))
Z_init[:,:] = np.asarray(results_R3_p['latent_params']['Z'])

In [ ]:
geometry = SphericalGeometry(d=2, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True, init_params = {"alpha0":0.0,"Z": Z_init})

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_S2_c.pkl")


alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]

#### With weights

In [ ]:
geometry = SphericalGeometry(d=2, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_S2_p.pkl")

## Models over $\mathbb{H}^d$

### Model $\mathbb{H}^1$

#### Without weights

In [ ]:
with open("Florentine_R1_c.pkl", "rb") as f:
    results_R1_c = pickle.load(f)

Z_init = np.zeros((15,2))
Z_init[:,0:1] = np.asarray(results_R1_c['latent_params']['Z'])
for i in range(15):
    Z_init[i] = geometry.project_to_domain(Z_init[i])

In [ ]:
geometry = HyperbolicGeometry(d=1, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True, init_params = {"alpha0":0.0,"Z": Z_init})

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_H1_c.pkl")

alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]

#### With weights

In [ ]:
geometry = HyperbolicGeometry(d=1, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_H1_p.pkl")

### Model $\mathbb{H}^2$

#### Without weights

In [ ]:
with open("Florentine_R2_p.pkl", "rb") as f:
    results_R2_p = pickle.load(f)

Z_init = np.zeros((n,3))
Z_init[:,0:2] = np.asarray(results_R2_p['latent_params']['Z'])

In [ ]:
geometry = HyperbolicGeometry(d=2, D=10.0)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"xi": np.ones(n)})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True, init_params = {"alpha0":0.0,"Z": Z_init})

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_H2_c.pkl")

alpha0 = model.inferred["MAP"]["final_params"]["alpha0"]
Z = model.inferred["MAP"]["final_params"]["Z"]

#### With weights

In [ ]:
geometry = HyperbolicGeometry(d=2, D=diam)
model = LatentSpaceModel(Y=Y, geometry=geometry, fixed_params={"Z": Z, "alpha0": alpha0})
inference = MAPInference(model, lr=lr_MAP)
result = inference.fit_multi_start(n_starts=n_starts_MAP, n_iter=n_iter_MAP, verbose=True, use_tqdm=True)

analysis = ModelAnalysis(model,inference_key="MAP")
analysis.plot_logposterior_trace()
analysis.compare_with_adjacency(analysis.P,title="Probability matrix")
analysis.internal_validation(n_sim=n_sim_iv)
analysis.plot_ppc_distributions()
analysis.laplacian_validation() 
analysis.binary_prediction_metrics()
analysis.plot_roc_curve()
analysis.plot_precision_recall_curve()
analysis.confusion_matrix(threshold=0.5)
analysis.cross_validation(n_folds=n_folds_cv, seed=seed_MAP, n_iter=n_iter_cv, lr=0.9*lr_MAP)
print(analysis.cross_validation_summary())

model.save("Florentine_H2_p.pkl")

## Results

### Visualization of latent spaces 

$\mathbb{R}^1$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
with open("Florentine_R1_p.pkl", "rb") as f:
    results = pickle.load(f)
z = np.asarray(results["latent_params"]["Z"]).ravel()
xi = np.asarray(results["latent_params"]["xi"]).ravel()
Y = np.asarray(results["Y"])
n = len(z)
perm = np.argsort(z)
z = z[perm]
xi = xi[perm]
Y = Y[np.ix_(perm, perm)]
labels = perm
y = np.arange(n)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
cmap = plt.cm.Blues_r
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(n):
    ax.axhline(y=i, color="lightgray", lw=1, zorder=0)
for i in range(n):
    for j in range(i + 1, n):
        if Y[i, j] > 0:
            ax.plot([z[i], z[j]], [y[i], y[j]], color="black", lw=1.2, alpha=0.6, zorder=1 )

sc = ax.scatter(z, y, c=xi, cmap=cmap, norm=norm, s=350, edgecolor="black", zorder=3)
for i in range(n):
    txt = ax.text(z[i], y[i], str(labels[i]), color="white", fontsize=10, fontweight="bold", ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Latent position")
ax.set_ylabel("Node")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
ax.set_xticks([])
ax.set_yticks([])
plt.show()

$\mathbb{R}^2$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
with open("Florentine_R2_p.pkl", "rb") as f:
    results_R2_p = pickle.load(f)
Z = np.asarray(results_R2_p['latent_params']['Z'])
xi = np.asarray(results_R2_p['latent_params']['xi']).ravel()
Y = np.asarray(results_R2_p['Y'])
n = len(xi)
x = Z[:, 0]
y = Z[:, 1]
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(n):
    for j in range(i + 1, n):
        if Y[i, j] > 0:
            ax.plot([x[i], x[j]], [y[i], y[j]], color="black", alpha=0.35, lw=1.0, zorder=1)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(x, y, c=xi, cmap="Blues_r", norm=norm, s=300, edgecolor="black", zorder=3)
for i in range(n):
    txt = ax.text(x[i], y[i], str(i), color="white", fontsize=10, fontweight="bold", ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_aspect("equal")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

$\mathbb{R}^3$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patheffects as pe
with open("Florentine_R3_p.pkl", "rb") as f:
    results_R3_p = pickle.load(f)
Z = np.asarray(results_R3_p["latent_params"]["Z"])
xi = np.asarray(results_R3_p["latent_params"]["xi"]).ravel()
Y = np.asarray(results_R3_p["Y"])
n = len(xi)
x = Z[:, 0]
y = Z[:, 1]
z = Z[:, 2]
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
for i in range(n):
    for j in range(i + 1, n):
        if Y[i, j] > 0:
            ax.plot([x[i], x[j]], [y[i], y[j]], [z[i], z[j]], color="black", alpha=0.6, lw=1.0, zorder=1)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(x, y, z, c=xi, cmap="Blues_r", norm=norm, s=300, edgecolors="black", depthshade=True, zorder=3, alpha=1.0)
for i in range(n):
    txt = ax.text(x[i], y[i], z[i], str(i), color="white", fontsize=10, fontweight="bold", ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax, shrink=0.75)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_box_aspect([1, 1, 1])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])
ax.grid(False)
ax.view_init(elev=20, azim=45)
plt.tight_layout()
plt.show()

$\mathbb{S}^1$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
with open("Florentine_S1_p.pkl", "rb") as f:
    results_S1_p = pickle.load(f)
Z  = np.asarray(results_S1_p["latent_params"]["Z"])
xi = np.asarray(results_S1_p["latent_params"]["xi"]).ravel()
Y  = np.asarray(results_S1_p["Y"])
n = len(xi)
theta = np.arctan2(Z[:,1], Z[:,0])
r0 = 1.0
dr = 0.5
radii = r0 + dr*np.arange(n)
x = radii*np.cos(theta)
y = radii*np.sin(theta)
fig, ax = plt.subplots(figsize=(10,8))
tt = np.linspace(0, 2*np.pi, 500)
for r in radii:
    ax.plot(r*np.cos(tt), r*np.sin(tt), color="lightgray", lw=1, zorder=0)
for i in range(n):
    for j in range(i+1,n):
        if Y[i,j] > 0:
            ax.plot([x[i],x[j]], [y[i],y[j]], color="black", alpha=0.35, lw=1.0, zorder=1)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(x, y, c=xi, cmap="Blues_r", norm=norm, s=300, edgecolor="black", zorder=3)
for i in range(n):
    txt = ax.text(x[i], y[i], str(i), color="white", fontsize=10, fontweight="bold", ha="center", va="center", zorder=4)
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(True)
plt.tight_layout()
plt.show()

$\mathbb{S}^2$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patheffects as pe

with open("Florentine_S2_p.pkl", "rb") as f:
    results_S2_p = pickle.load(f)
Z = np.asarray(results_S2_p["latent_params"]["Z"])
xi = np.asarray(results_S2_p["latent_params"]["xi"]).ravel()
Y = np.asarray(results_S2_p["Y"])
n = len(xi)
x = Z[:, 0]
y = Z[:, 1]
z = Z[:, 2]
R = (x[0]**2 + y[0]**2  + z[0]**2)**0.5
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
u = np.linspace(0, 2*np.pi, 100)
v = np.linspace(0, np.pi, 100)
xs = R*np.outer(np.cos(u), np.sin(v))
ys = R*np.outer(np.sin(u), np.sin(v))
zs = R*np.outer(np.ones_like(u), np.cos(v))
ax.plot_surface(xs, ys, zs, alpha=0.08, color="gray", linewidth=0, zorder=0)
for i in range(n):
    for j in range(i + 1, n):
        if Y[i, j] > 0:
            ax.plot([x[i], x[j]], [y[i], y[j]], [z[i], z[j]], color="black", alpha=0.25, lw=1.0, zorder=1)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(x, y, z, c=xi, cmap="Blues_r", norm=norm, s=250, edgecolors="black", depthshade=True, zorder=3, alpha=0.85)
for i in range(n):
    txt = ax.text(x[i], y[i], z[i], str(i), color="white", fontsize=10, fontweight="bold", ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax, shrink=0.75)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_box_aspect([1,1,1])
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")
ax.grid(False)
plt.tight_layout()
plt.show()

$\mathbb{H}^1$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
with open("Florentine_H1_p.pkl", "rb") as f:
    results_H1_p = pickle.load(f)
diam = 10.0
R = diam / np.pi
Z = np.asarray(results_H1_p["latent_params"]["Z"])
xi = np.asarray(results_H1_p["latent_params"]["xi"]).ravel()
Y = np.asarray(results_H1_p["Y"])
n = len(xi)
x0 = Z[:, 0]
x1 = Z[:, 1]
t_nodes = np.arcsinh(x0 / R)
perm = np.argsort(t_nodes)
t_nodes = t_nodes[perm]
xi = xi[perm]
Y = Y[np.ix_(perm, perm)]
labels = perm
lim_range_t = max(np.max(np.abs(t_nodes))*1.15, np.pi/3)
t = np.linspace(-lim_range_t, lim_range_t, 1000)
base_x = R*np.sinh(t)
base_y = R*np.cosh(t)
spacing = 2.0
node_x = np.zeros(n)
node_y = np.zeros(n)
fig, ax = plt.subplots(figsize=(12,10))
for i in range(n):
    offset = -i*spacing
    ax.plot(base_x, base_y + offset, color="lightgray", lw=1.0, zorder=0)
    node_x[i] = R*np.sinh(t_nodes[i])
    node_y[i] = R*np.cosh(t_nodes[i]) + offset
for i in range(n):
    for j in range(i+1, n):
        if Y[i,j] > 0:
            ax.plot([node_x[i], node_x[j]], [node_y[i], node_y[j]], color="black", alpha=0.35, lw=1.0, zorder=1)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(node_x, node_y, c=xi, cmap="Blues_r", norm=norm, s=300, edgecolor="black", zorder=3)
for i in range(n):
    txt = ax.text(node_x[i], node_y[i], str(labels[i]), color="white", fontsize=10, fontweight="bold",ha="center",va="center",zorder=4)
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

$\mathbb{H}^2$

In [ ]:

import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patheffects as pe
with open("Florentine_H2_p.pkl", "rb") as f:
    results_H2_p = pickle.load(f)
Z = np.asarray(results_H2_p["latent_params"]["Z"])
xi = np.asarray(results_H2_p["latent_params"]["xi"]).ravel()
Y = np.asarray(results_H2_p["Y"])
n = len(xi)
x0 = Z[:, 0]
x1 = Z[:, 1]
x2 = Z[:, 2]
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")
rmax = np.max(np.sqrt(x1**2 + x2**2)) * 0.75
r = np.linspace(0, rmax, 100)
theta = np.linspace(0, 2*np.pi, 150)
R, Theta = np.meshgrid(r, theta)
X1 = R * np.cos(Theta)
X2 = R * np.sin(Theta)
X0 = np.sqrt((diam/np.pi)**2 + X1**2 + X2**2)
ax.plot_surface(X1, X2, X0, alpha=0.08, color="gray", linewidth=0, antialiased=True)
for i in range(n):
    for j in range(i + 1, n):
        if Y[i, j] > 0:
            ax.plot([x0[i], x0[j]], [x1[i], x1[j]],  [x2[i], x2[j]], color="black", alpha=0.25, lw=1.0)
norm = Normalize(vmin=np.min(xi), vmax=np.max(xi))
sc = ax.scatter(x0, x1, x2, c=xi, cmap="Blues_r", norm=norm, s=250, edgecolors="black", depthshade=True)
for i in range(n):
    txt = ax.text(x0[i], x1[i], x2[i], str(i), color="white", fontsize=10, fontweight="bold", ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=2, foreground="black")])
cbar = plt.colorbar(sc, ax=ax, shrink=0.75)
cbar.set_label(r"$\xi^{(i)}$")
ax.set_box_aspect([1,1,1])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])
ax.grid(False)
ax.view_init(elev=15, azim=30)
plt.tight_layout()
plt.show()

### Visualization of probability matrices

### Model statistics

### Internal validation

### External validation